# LSTM Gesture Classifier

This notebook trains a **stacked LSTM** (Long Short-Term Memory) model to classify
hand gestures from time-series sensor data recorded by a sensory data glove.

**Data format:**  
Each gesture trial is a CSV file (~5 s, ~31 Hz) stored in a folder whose name is the
gesture label.  
Column naming convention: `{hand}_{segment}_{loc}_{channel}`  
e.g. `left_thumb_mid_ax`, `right_index_prox_pitch`, `left_thumb_flex_mcp`

**Pipeline overview:**
1. Select sensor channels via config flags
2. Load + filter + resample + normalise all trials
3. Stratified train / val / test split
4. Build and train a stacked LSTM with early stopping
5. Evaluate with confusion matrix and per-class classification report
6. Save model weights and a JSON results summary

---
**Column naming convention:**
```
{hand}_{segment}_{loc}_{channel}

Examples:
  left_thumb_mid_ax          right_index_prox_pitch
  left_thumb_flex_mcp        right_palm_az
  left_wrist_roll
```
Quaternion columns (`qw`, `qx`, `qy`, `qz`) are **always excluded**.


---
## Cell 1 — Install Dependencies

This cell ensures all required Python packages are installed in the current
environment before any imports are attempted.  It is safe to re-run — it will
simply confirm the packages are already present.


In [ ]:
import subprocess, sys

# List of third-party libraries this project depends on.
# TensorFlow provides the LSTM and training infrastructure;
# scikit-learn provides metrics (confusion matrix, classification report);
# pandas/numpy handle data tables and array maths;
# scipy provides signal-processing filters;
# matplotlib/seaborn handle all plots.
pkgs = [
    'tensorflow',
    'scikit-learn',
    'pandas',
    'numpy',
    'scipy',
    'matplotlib',
    'seaborn',
]

# Call pip programmatically so the user doesn't have to leave the notebook.
# sys.executable ensures we install into the *same* Python that is running
# this kernel, not some other Python on the machine.
# '--quiet' suppresses the verbose pip output to keep the cell tidy.
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs
)
print('All dependencies installed.')


---
## Cell 2 — Imports

All standard library, third-party, and local imports are collected here.
Running this cell before Cell 3 (CONFIG) would fail because CONFIG variables
like `USE_LEFT_HAND` are not yet defined — the **actual imports are in Cell 5**
(Load Dataset).  This cell imports only the utility helpers needed to resolve
the sensor-column names that depend on the config flags.


In [ ]:
import sys, os

# ─── Make the project root importable ─────────────────────────────────────────
# The notebook lives in  ThesisRepo/ML/;  the utils/ package lives in
# ThesisRepo/utils/.  We add the parent directory to Python's module search
# path so that  `from utils.data_loader import ...`  works regardless of where
# Jupyter was launched from.
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

# ─── Local utilities ──────────────────────────────────────────────────────────
# build_column_groups : assembles the list of CSV column names to use as
#   model features, given which sensor groups are toggled on in CONFIG.
# HANDS, FINGERS, LOCS : constants listing all valid hand sides, finger names,
#   and IMU mounting locations in the glove hardware.
# IMU_ALL_CHANNELS, FLEX_CHANNELS, WRIST_ALL_CHANNELS : per-sensor channel name
#   lists (e.g. ['ax','ay','az','yaw','pitch','roll']).
from utils.data_loader import (
    build_column_groups, HANDS, FINGERS, LOCS,
    IMU_ALL_CHANNELS, FLEX_CHANNELS, WRIST_ALL_CHANNELS,
)


---
## Cell 3 — CONFIG

> **Edit all experiment settings here.**  Every tunable parameter lives in this
> cell — no other cell needs to be modified for standard experiments.

This is the single source of truth for the entire pipeline.  Variables defined
here are referenced by every subsequent cell.  Changing a value here and
re-running the whole notebook from Cell 4 onwards will reproduce a fully
self-consistent experiment.


In [ ]:
# =============================================================================
# CONFIGURATION  —  Edit this cell to control every aspect of the pipeline
# =============================================================================

# ── DATA ──────────────────────────────────────────────────────────────────────
DATA_ROOT = '../data'   # Root folder containing the 4 gesture label subfolders:
                        #   TwoHand_L_Fist_R_Fist_filtered_butterworth_lp/
                        #   TwoHand_L_Flat_R_Flat_filtered_butterworth_lp/
                        #   TwoHand_L_Okay_R_Okay_filtered_butterworth_lp/
                        #   TwoHand_L_Rock_R_Rock_filtered_butterworth_lp/

FS_HZ      = 31.0       # Confirmed sampling rate (~156 rows / 5 s = 31.2 Hz)
TARGET_LEN = 156        # Native timesteps per trial (5 s × ~31 Hz); set lower
                        # (e.g. 78) to halve resolution and speed up training.

# ── COLUMN / SENSOR SELECTION ─────────────────────────────────────────────────
# Toggle entire sensor groups on / off.
# All combinations are valid — the feature list is built automatically.

USE_LEFT_HAND  = True    # Include left-hand sensors
USE_RIGHT_HAND = True    # Include right-hand sensors

# Per-finger IMU locations
USE_DISTAL_IMU   = True  # 'mid'  — distal phalanx IMU  (tip segment)
USE_PROXIMAL_IMU = True  # 'prox' — proximal phalanx IMU (base segment)

# IMU channel types (applies to finger IMUs AND palm_prox AND wrist)
USE_ACCELEROMETER = True  # ax, ay, az
USE_ORIENTATION   = True  # yaw, pitch, roll  (wrist uses 'heading' instead of 'yaw')

# Flex sensors  (MCP and PIP joints per finger)
USE_FLEX_SENSORS  = True  # {hand}_{finger}_{mcp_flex / pip_flex}
                           # NOTE: palm flex is always -1 (no sensor) → auto-excluded

# Back-of-hand IMU  (palm_prox — the real palm IMU; palm_mid is always zero → auto-excluded)
USE_PALM_IMU  = True

# Wrist IMU  ({hand}_wrist_{ax/ay/az/heading/pitch/roll})
USE_WRIST_IMU = True

# ── PREPROCESSING ─────────────────────────────────────────────────────────────
# Temporal filter applied to each channel before training.
# The data files are already Butterworth LP-filtered at 5 Hz by the glove firmware;
# set 'none' to use them as-is, or apply an additional filter here.
#
# Options: 'none' | 'butterworth_lp' | 'butterworth_hp' | 'butterworth_bp'
#          'moving_average' | 'savgol' | 'median'
FILTER_TYPE = 'none'

# Normalization applied after resampling
# Options: 'minmax' (→ [0,1]) | 'zscore' (zero-mean unit-var) | 'none'
NORMALIZATION   = 'minmax'
PER_SAMPLE_NORM = True   # True = normalize each trial independently (recommended)

# ── DATASET SPLIT ─────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70   # 70 % train   →  280 / 400 samples
VAL_RATIO   = 0.10   # 10 % val     →   40 / 400 samples
                     # 20 % test    →   80 / 400 samples (auto)
RANDOM_SEED = 42

# ── MODEL HYPERPARAMETERS ─────────────────────────────────────────────────────
LSTM_UNITS        = [128, 64]  # Units per stacked LSTM layer
DROPOUT_RATE      = 0.3        # Dropout after each LSTM layer
RECURRENT_DROPOUT = 0.2        # Recurrent dropout within LSTM cells
DENSE_UNITS       = [64]       # Hidden FC units before softmax output

# ── TRAINING ──────────────────────────────────────────────────────────────────
EPOCHS              = 80
BATCH_SIZE          = 32
LEARNING_RATE       = 1e-3
EARLY_STOP_PATIENCE = 15


---
## Cell 4 — Build Feature Column List from Config

Using the boolean flags defined in CONFIG, this cell constructs the explicit list
of CSV column names (`feature_cols`) that will be fed to the LSTM as input features.
It also prints a grouped summary so you can verify which sensor channels are active
before loading any data.


In [ ]:
# ─── Resolve which hands to include ──────────────────────────────────────────
# HANDS is a constant list ['left', 'right'] defined in data_loader.
# We filter it down based on the USE_LEFT_HAND / USE_RIGHT_HAND flags from CONFIG.
# The  `or HANDS`  fallback means: if both flags were False, include both anyway
# (prevents an accidental empty-feature situation).
_hands = ([h for h in HANDS if (USE_LEFT_HAND  and h == "left")
                              or (USE_RIGHT_HAND and h == "right")]
          or HANDS)

# All five fingers are always included — only the IMU location (proximal vs.
# distal) is selectable via USE_DISTAL_IMU / USE_PROXIMAL_IMU.
_fingers = FINGERS

# ─── Resolve which IMU mounting locations to include ─────────────────────────
# 'mid'  = distal phalanx (fingertip segment)
# 'prox' = proximal phalanx (knuckle segment)
# Same `or LOCS` fallback as above.
_locs = [l for l in LOCS
         if (USE_DISTAL_IMU   and l == "mid")
         or (USE_PROXIMAL_IMU and l == "prox")] or LOCS

# ─── Assemble the full column-name list ──────────────────────────────────────
# build_column_groups() returns a flat list of strings like
# ['left_thumb_mid_ax', 'left_thumb_mid_ay', ..., 'right_wrist_roll']
# This list tells the data loader which CSV columns to extract, and its length
# becomes NUM_FEATURES — the width of each timestep vector fed to the LSTM.
feature_cols = build_column_groups(
    hands              = _hands,
    fingers            = _fingers,
    locs               = _locs,
    include_flex       = USE_FLEX_SENSORS,   # bend-angle sensors at finger joints
    include_palm_prox  = USE_PALM_IMU,       # back-of-hand IMU
    include_wrist      = USE_WRIST_IMU,      # wrist IMU
    include_accel      = USE_ACCELEROMETER,  # ax, ay, az channels
    include_orient     = USE_ORIENTATION,    # yaw/pitch/roll channels
)

print(f"Feature columns selected: {len(feature_cols)}")
print()

# ─── Print a grouped summary for visual verification ─────────────────────────
# Columns share a common 3-part prefix (hand_segment_location).
# This loop groups them by prefix and prints up to 12 groups.
groups = {}
for c in feature_cols:
    parts = c.split("_")
    grp   = "_".join(parts[:3]) if len(parts) >= 4 else "_".join(parts[:2])
    groups.setdefault(grp, []).append(c)

for grp, cols in list(groups.items())[:12]:
    print(f"  {grp:40s}  ({len(cols)} cols): {[c.split('_')[-1] for c in cols]}")
if len(groups) > 12:
    print(f"  ... and {len(groups)-12} more groups")


---
## Cell 5 — Load and Preprocess Dataset

This cell performs all remaining imports, then calls `build_dataset()` to walk
every gesture subfolder, load the CSV trial files, apply optional filtering and
resampling, and stack everything into two NumPy arrays: `X` (the 3-D sensor
tensor) and `y` (the integer class labels).  It also plots the class distribution
so you can spot any imbalance before training.


In [ ]:
# ─── Standard library and third-party imports ─────────────────────────────────
# These are imported here (rather than at the top of Cell 2) because they are
# not needed to build the feature-column list, and some (tensorflow) take a few
# seconds to import — deferring them keeps the early cells fast.

from utils.data_loader import build_dataset, split_dataset
import numpy as np                        # fast numerical arrays (like MATLAB matrices)
import matplotlib.pyplot as plt           # plotting (learning curves, confusion matrix)
import seaborn as sns                     # nicer statistical plots (heatmap for CM)

from tensorflow import keras              # high-level model API
from tensorflow.keras import layers       # LSTM, Dense, Dropout layer classes
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
                                          # training callbacks (explained in Cell 8)
from tensorflow.keras.utils import to_categorical  # integer label → one-hot vector
from sklearn.metrics import classification_report, confusion_matrix
                                          # precision/recall/F1 and confusion matrix
                                          # (both covered in your COMP3308 lectures)
from pathlib import Path                  # cross-platform file-path handling
import json                               # save results as a human-readable JSON file
from datetime import datetime             # generate a timestamp for unique file names

# ─── Timestamp ────────────────────────────────────────────────────────────────
# Every output file (plots, model weights, results JSON) gets a timestamp suffix
# so multiple experiment runs never overwrite each other.
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

# ─── Load and preprocess every trial ─────────────────────────────────────────
# build_dataset() does the following for every CSV file it finds under DATA_ROOT:
#   1. Read the CSV and keep only the columns in feature_cols.
#   2. Optionally apply a temporal filter (e.g. Butterworth low-pass) to smooth
#      each sensor channel over time — controlled by FILTER_TYPE in CONFIG.
#   3. Resample every trial to exactly TARGET_LEN time steps so that all trials
#      have the same length (the LSTM requires fixed-length inputs).
#   4. Optionally normalise the values — see NORMALIZATION / PER_SAMPLE_NORM in
#      CONFIG.  Normalisation is the same min-max or z-score technique from your
#      COMP3308 lectures; it puts all sensor channels on the same numerical scale
#      so that channels with large absolute values (e.g. accelerometer) don't
#      dominate channels with small values (e.g. flex sensors).
#
# Returns:
#   X            : shape (N, TARGET_LEN, NUM_FEATURES)  — the input tensor
#                    N           = total number of trials across all classes
#                    TARGET_LEN  = timesteps per trial (set in CONFIG)
#                    NUM_FEATURES = number of sensor channels (= len(feature_cols))
#   y            : shape (N,)  — integer class index for each trial (0,1,2,3)
#   CLASS_NAMES  : list of human-readable gesture names, e.g. ['Fist','Flat',...]
#   feature_cols_used : the subset of feature_cols that were actually present in
#                       the CSV files (may differ if a sensor was disconnected)
X, y, CLASS_NAMES, feature_cols_used = build_dataset(
    data_root       = DATA_ROOT,
    feature_cols    = feature_cols,
    filter_type     = FILTER_TYPE,
    fs              = FS_HZ,
    target_len      = TARGET_LEN,
    normalization   = NORMALIZATION,
    per_sample_norm = PER_SAMPLE_NORM,
)

# ─── Derived constants ────────────────────────────────────────────────────────
NUM_CLASSES  = len(CLASS_NAMES)    # number of output neurons in the final layer
NUM_FEATURES = X.shape[2]          # number of input features per timestep
                                   # (may be < len(feature_cols) if some columns
                                   #  were missing from the CSV files)

# ─── Class distribution bar chart ────────────────────────────────────────────
# Before training it is important to check that each class has roughly the same
# number of trials.  A heavily imbalanced dataset can cause the model to achieve
# high accuracy simply by always predicting the majority class — a classic pitfall
# discussed in your COMP3308 lectures.
fig, ax = plt.subplots(figsize=(8, 4))
unique, counts = np.unique(y, return_counts=True)  # find unique class indices and how often each appears
ax.bar([CLASS_NAMES[i] for i in unique], counts, color='steelblue', edgecolor='black')
ax.set_xlabel('Gesture Class')
ax.set_ylabel('Number of Trials')
ax.set_title('Class Distribution')
plt.tight_layout()

import os; os.makedirs('results', exist_ok=True)  # create output folder if absent
plt.savefig(f'results/class_distribution_{TIMESTAMP}.png', dpi=150)
plt.show()

# ─── Summary printout ─────────────────────────────────────────────────────────
# X.shape is (N, T, C): N samples, T timesteps, C feature channels.
# Think of each sample as a short video clip of hand movement;
# each "frame" (timestep) contains C sensor readings.
print(f'Input shape: X={X.shape}  y={y.shape}')
print(f'Classes: {CLASS_NAMES}')


---
## Cell 6 — Train / Val / Test Split

We divide the dataset into three non-overlapping subsets — the same principle
taught in your COMP3308 lectures (holdout procedure):

| Subset | Fraction | Purpose |
|--------|----------|---------|
| **Train** | 70 % | The model *sees* these examples and updates its weights |
| **Val**   | 10 % | Monitored after each epoch to detect overfitting without touching test data |
| **Test**  | 20 % | Evaluated exactly **once** at the end to report unbiased performance |

The split is **stratified**: each subset contains the same proportion of each
class.  Labels are then **one-hot encoded** (converted from a single integer to a
vector of zeros with a 1 in the correct position) so that Keras can compute
categorical crossentropy loss.


In [ ]:
"""
Stratified split into train / val / test subsets.
Labels are one-hot encoded for Keras categorical crossentropy.
"""

# Create output directories now so later cells can save files without checking.
os.makedirs('results', exist_ok=True)
os.makedirs('saved_models', exist_ok=True)

# ─── Stratified split ─────────────────────────────────────────────────────────
# split_dataset() shuffles the data with RANDOM_SEED, then carves off
# TRAIN_RATIO for training and VAL_RATIO for validation; the remainder becomes
# the test set.  "Stratified" means it preserves the class proportions in each
# split — essential when one class has fewer trials than others.
# This is the holdout procedure your lectures describe.
(X_train, y_train_int), (X_val, y_val_int), (X_test, y_test_int) = split_dataset(
    X, y,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = RANDOM_SEED,
)

# ─── One-hot encode labels ────────────────────────────────────────────────────
# Your COMP3308 lectures describe softmax output for multi-class classification.
# The categorical crossentropy loss function needs the *target* expressed as a
# probability vector of the same shape as the model output.
#
# For example, with 4 gesture classes:
#   integer label  2   →   one-hot vector  [0, 0, 1, 0]
#
# to_categorical(2, num_classes=4) does exactly this conversion.
# After encoding, y_train has shape (N_train, 4), y_val (N_val, 4), etc.
y_train = to_categorical(y_train_int, num_classes=NUM_CLASSES)
y_val   = to_categorical(y_val_int,   num_classes=NUM_CLASSES)
y_test  = to_categorical(y_test_int,  num_classes=NUM_CLASSES)

# ─── Print shapes to verify ───────────────────────────────────────────────────
print('Split shapes:')
print(f'  X_train : {X_train.shape}   y_train : {y_train.shape}')
print(f'  X_val   : {X_val.shape}     y_val   : {y_val.shape}')
print(f'  X_test  : {X_test.shape}    y_test  : {y_test.shape}')

# ─── Verify class balance in each split ───────────────────────────────────────
# A well-stratified split should show similar counts per class in all three sets.
# If one class is under-represented in the test set, the reported accuracy may be
# misleading — this printout lets you catch that immediately.
for split_name, split_y in [('Train', y_train_int), ('Val', y_val_int), ('Test', y_test_int)]:
    u, c = np.unique(split_y, return_counts=True)  # unique class indices and their counts
    balance = ', '.join(f'{CLASS_NAMES[i]}:{n}' for i, n in zip(u, c))
    print(f'  {split_name:6s} class counts: {balance}')


---
## Cell 7 — Model Definition

### What is an LSTM?

In your COMP3308 lectures you studied **feedforward networks**: each layer
receives the full input at once and has no memory of previous inputs.  An LSTM
is a **recurrent** network — it processes the input *one timestep at a time*,
maintaining a **hidden state** `h_t` that carries information from earlier
timesteps forward.  This makes it naturally suited to time-series data such as a
5-second glove recording.

At each timestep `t` the LSTM receives:
- `x_t` — the current sensor readings (one row of the data matrix, shape `[NUM_FEATURES]`)
- `h_{t-1}` — the hidden state from the previous timestep (the short-term memory)
- `c_{t-1}` — the cell state (the long-term memory)

It outputs an updated `h_t` and `c_t`.  Three learned **gates** control what
information is kept, written, or discarded:

| Gate | Function |
|------|----------|
| **Forget gate** | A sigmoid neuron (0 → forget, 1 → keep) that decides which parts of `c_{t-1}` to retain |
| **Input gate** | Decides what new information from `x_t` to write into the cell state |
| **Output gate** | Decides which part of the updated cell state to expose as `h_t` |

All gate weights are learned by **backpropagation** — the same algorithm from
your lectures, just applied through time (called Backpropagation Through Time,
BPTT).

### Why stack two LSTM layers?

The first LSTM reads raw sensor values and learns low-level temporal patterns
(e.g. "the wrist is accelerating upward").  The second LSTM receives those
patterns as input and learns higher-level patterns (e.g. "after the wrist rises,
the fingers curl — that's a Fist gesture").  Stacking is analogous to stacking
convolutional layers in the CNNs you studied.

### Architecture

```
Input (156 timesteps × NUM_FEATURES channels)
  → LSTM(128 units, return_sequences=True)  → Dropout(0.3)
  → LSTM(64  units, return_sequences=False) → Dropout(0.3)
  → Dense(64, activation='relu')            → Dropout(0.3)
  → Dense(NUM_CLASSES, activation='softmax')   ← predicted class probabilities
```


In [ ]:
"""
Build a stacked LSTM model.

Architecture
------------
Input  →  [LSTM(units, return_sequences=True)  →  Dropout]  × (n-1 layers)
       →   LSTM(units, return_sequences=False) →  Dropout
       →  [Dense(units, activation='relu')     →  Dropout]  × n_dense
       →   Dense(NUM_CLASSES, activation='softmax')

Notes
-----
- return_sequences=True is required for all LSTM layers except the last
  so that the next LSTM layer receives a full time-step sequence.
- recurrent_dropout > 0 disables CuDNN kernel; use 0.0 for GPU speed.
"""

import tensorflow as tf  # import here in case this cell is run in isolation

# ─── Seed both TensorFlow and NumPy for reproducibility ───────────────────────
# Neural network weights are initialised randomly.  Fixing the seed means two
# identical runs produce the same model, making experiments reproducible.
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


def build_lstm_model(
    input_shape,
    num_classes,
    lstm_units,
    dropout_rate,
    recurrent_dropout,
    dense_units,
):
    """
    Construct a configurable stacked LSTM classifier.

    Parameters
    ----------
    input_shape       : (timesteps, features)
    num_classes       : number of gesture classes (output units)
    lstm_units        : list[int] — units per LSTM layer
    dropout_rate      : float in [0, 1) — output dropout after each LSTM
    recurrent_dropout : float in [0, 1) — recurrent dropout inside LSTM
    dense_units       : list[int] — units per hidden FC layer

    Returns
    -------
    keras.Model
    """

    # ─── Define the input ─────────────────────────────────────────────────────
    # keras.Input specifies the shape of ONE sample (no batch dimension).
    # shape = (TARGET_LEN, NUM_FEATURES) means:
    #   TARGET_LEN  timesteps (rows in the CSV — one per ~32 ms sensor reading)
    #   NUM_FEATURES channels per timestep (one value per sensor channel)
    inputs = keras.Input(shape=input_shape, name='glove_input')
    x = inputs  # 'x' is a symbolic tensor that gets transformed by each layer

    # ── Stacked LSTM layers ────────────────────────────────────────────────────
    for i, units in enumerate(lstm_units):
        # return_sequences controls what the LSTM layer outputs:
        #
        #   True  → output h_t at EVERY timestep → shape (T, units)
        #           Required for all layers except the last, because the next
        #           LSTM layer needs a full sequence to process.
        #
        #   False → output ONLY the final h_T    → shape (units,)
        #           Used for the last LSTM layer; this single vector is a
        #           compressed summary of the entire 5-second gesture, which
        #           is then fed to the Dense classification head.
        return_seq = (i < len(lstm_units) - 1)  # True for all but the last layer

        x = layers.LSTM(
            units,
            return_sequences  = return_seq,
            # dropout applies to the input-to-gate connections (x_t → gates).
            # During each training pass, a fraction `dropout_rate` of input
            # values is randomly zeroed.  This is regularisation (reduces
            # overfitting), which your COMP3308 lectures cover.  At test time
            # Keras disables dropout automatically.
            dropout           = dropout_rate,
            # recurrent_dropout applies to the hidden-state-to-gate connections
            # (h_{t-1} → gates).  Same idea as dropout, but on the recurrent
            # (memory) path rather than the input path.
            recurrent_dropout = recurrent_dropout,
            name              = f'lstm_{i+1}',
        )(x)

        # An additional Dropout layer after the LSTM output.
        # This zeros a further fraction of h_t values before passing them to the
        # next layer — a second line of defence against overfitting.
        x = layers.Dropout(dropout_rate, name=f'dropout_lstm_{i+1}')(x)

    # ── Fully-connected head ───────────────────────────────────────────────────
    # After the last LSTM, x has shape (units,) — a flat vector summarising the
    # gesture.  We add one or more Dense (fully-connected) layers to learn a
    # non-linear mapping from this summary to class probabilities.
    # This is exactly the multi-layer perceptron structure from your lectures.
    for j, units in enumerate(dense_units):
        # ReLU activation: output = max(0, weighted_sum + bias).
        # From your lectures: ReLU avoids the vanishing-gradient problem that
        # afflicts sigmoid activations in deep networks.
        x = layers.Dense(units, activation='relu', name=f'dense_{j+1}')(x)
        # Dropout after each Dense layer — same regularisation rationale as above.
        x = layers.Dropout(dropout_rate, name=f'dropout_dense_{j+1}')(x)

    # ── Output layer ──────────────────────────────────────────────────────────
    # num_classes neurons, one per gesture class.
    # softmax converts the raw neuron outputs into a valid probability
    # distribution: all values are in [0, 1] and sum to 1.
    # The predicted class is whichever neuron has the highest probability.
    # This is the same softmax from your COMP3308 lectures on multi-class
    # classification.
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    # Package inputs and outputs into a Keras Model object.
    return keras.Model(inputs, outputs, name='LSTM_Gesture_Classifier')


# ─── Instantiate the model ────────────────────────────────────────────────────
# We pass (TARGET_LEN, NUM_FEATURES) as input_shape.  These were derived from
# the dataset in Cell 5.  Every hyperparameter comes from CONFIG (Cell 3).
model = build_lstm_model(
    input_shape       = (TARGET_LEN, NUM_FEATURES),
    num_classes       = NUM_CLASSES,
    lstm_units        = LSTM_UNITS,       # e.g. [128, 64] → two LSTM layers
    dropout_rate      = DROPOUT_RATE,
    recurrent_dropout = RECURRENT_DROPOUT,
    dense_units       = DENSE_UNITS,      # e.g. [64] → one hidden Dense layer
)

# Print a layer-by-layer summary showing shapes and parameter counts.
# "Param #" tells you how many weights each layer has; the LSTM has far more
# parameters than a Dense layer of the same width because it has four sets of
# gate weights (forget, input, output, and cell-candidate gates).
model.summary()


---
## Cell 8 — Training

This cell compiles the model (attaches the loss function and optimiser), sets up
two training callbacks, then runs the training loop.  After training, it plots
the **learning curves** — loss and accuracy on both the training and validation
sets across epochs.

### The Adam optimiser
Your COMP3308 lectures describe gradient descent: update each weight by
`w ← w − η · ∂L/∂w`, where `η` is the learning rate.  Adam is a smarter
variant that maintains a *per-weight adaptive learning rate*.  Weights whose
gradients have been consistently large get a smaller effective step size (to
avoid overshooting); weights whose gradients have been small get a larger step
(to converge faster).  In practice Adam converges faster and is less sensitive
to the choice of `LEARNING_RATE` than plain gradient descent.

### EarlyStopping
Your lectures explain that training too long causes overfitting — the model
memorises the training set and loses the ability to generalise.  EarlyStopping
automates the decision: it monitors `val_loss` after every epoch, and if it has
not improved for `EARLY_STOP_PATIENCE` epochs in a row, training stops and the
best weights are automatically restored.  No need to guess the right epoch count.

### ReduceLROnPlateau
If `val_loss` stops improving for 5 consecutive epochs (a "plateau"), the
learning rate is halved.  Think of it as switching from large steps to fine-
grained steps when you are close to the goal — the optimiser can then find a
better minimum without overshooting.


In [ ]:
"""
Compile and train the LSTM model.

Callbacks
---------
EarlyStopping     — restores best weights; stops after EARLY_STOP_PATIENCE
                    epochs of no improvement in val_loss.
ReduceLROnPlateau — halves the learning rate when val_loss plateaus for 5
                    epochs; minimum LR = 1e-6.
"""

# ── Compile ────────────────────────────────────────────────────────────────────
# .compile() does not run any computation; it just attaches the loss function,
# optimiser, and evaluation metrics to the model so Keras knows how to train it.
model.compile(
    # Adam: adaptive gradient descent — see the markdown above for an explanation.
    # learning_rate sets the initial step size; Adam will adjust it per weight.
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE),

    # categorical_crossentropy: the standard loss for multi-class softmax output.
    # For a sample whose true label is class 2 (one-hot [0,0,1,0]) and the model
    # predicts [0.1, 0.1, 0.7, 0.1], loss = -log(0.7) ≈ 0.36.
    # Perfect prediction (probability 1.0 on the correct class) → loss = 0.
    # This is the multi-class extension of binary cross-entropy from your lectures.
    loss      = 'categorical_crossentropy',

    # Report accuracy at the end of each epoch so we can plot it.
    # Accuracy = fraction of samples where argmax(predicted) == true label.
    metrics   = ['accuracy'],
)

# ── Callbacks ──────────────────────────────────────────────────────────────────
# Callbacks are functions that Keras calls automatically at key points during
# training (end of each epoch, end of each batch, etc.).
callbacks = [
    EarlyStopping(
        monitor              = 'val_loss',        # watch validation loss
        patience             = EARLY_STOP_PATIENCE,  # stop if no improvement for this many epochs
        restore_best_weights = True,              # rewind to the best checkpoint automatically
        verbose              = 1,                 # print a message when triggered
    ),
    ReduceLROnPlateau(
        monitor  = 'val_loss',   # same signal as EarlyStopping
        factor   = 0.5,          # multiply the learning rate by 0.5 (halve it)
        patience = 5,            # wait 5 plateau epochs before reducing
        min_lr   = 1e-6,         # never reduce below this floor
        verbose  = 1,
    ),
]

# ── Train ──────────────────────────────────────────────────────────────────────
print(f'Training on {len(X_train)} samples  |  '
      f'Validating on {len(X_val)} samples  |  '
      f'Max epochs: {EPOCHS}')

# model.fit() runs the main training loop:
#   For each epoch (up to EPOCHS):
#     1. Shuffle the training data.
#     2. Split it into mini-batches of size BATCH_SIZE.
#     3. For each batch: forward pass → compute loss → backpropagate → update weights.
#     4. Evaluate on validation data (no weight updates — read-only).
#     5. Call all callbacks (EarlyStopping, ReduceLROnPlateau).
# `history` stores the loss and accuracy curves for plotting below.
history = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),   # evaluated after each epoch, no training
    epochs          = EPOCHS,            # maximum number of complete passes through the data
    batch_size      = BATCH_SIZE,        # number of samples per gradient update step
    callbacks       = callbacks,         # hook in EarlyStopping and ReduceLROnPlateau
    verbose         = 1,                 # print one line per epoch to the output
)

# ── Learning curves ────────────────────────────────────────────────────────────
# Plot training vs. validation loss and accuracy over epochs.
# What to look for:
#   GOOD: both curves converge together and level off at similar values.
#   OVERFITTING: train loss keeps falling while val loss starts rising.
#   UNDERFITTING: both curves are high / still decreasing when training stops.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_ran = range(1, len(history.history['loss']) + 1)  # actual epoch count (may be < EPOCHS if early stopped)

# ─── Left panel: Loss ─────────────────────────────────────────────────────────
axes[0].plot(epochs_ran, history.history['loss'],     label='Train loss',
             color='#20808D', linewidth=2)
axes[0].plot(epochs_ran, history.history['val_loss'], label='Val loss',
             color='#A84B2F', linewidth=2, linestyle='--')
axes[0].set_title('Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Categorical Crossentropy')   # the loss value being minimised
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ─── Right panel: Accuracy ────────────────────────────────────────────────────
axes[1].plot(epochs_ran, history.history['accuracy'],     label='Train acc',
             color='#20808D', linewidth=2)
axes[1].plot(epochs_ran, history.history['val_accuracy'], label='Val acc',
             color='#A84B2F', linewidth=2, linestyle='--')
axes[1].set_title('Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training History — LSTM Gesture Classifier', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'results/training_curves_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Training complete. Best val_loss: '
      f'{min(history.history["val_loss"]):.4f}  |  '
      f'Best val_acc: {max(history.history["val_accuracy"]):.4f}')


---
## Cell 9 — Evaluation

We now evaluate the trained model on the **held-out test set** — data the model
has *never seen* during training or validation.  This gives an unbiased estimate
of how well the model will perform on genuinely new gestures.

Three outputs are produced:

1. **Overall test accuracy and loss** — single numbers summarising performance.
2. **Classification report** — per-class precision, recall, and F1 score, as
   taught in your COMP3308 lectures.
3. **Confusion matrix** — a heatmap showing which gestures the model confuses
   with which others.  Rows = true labels, columns = predicted labels.  A
   perfect classifier has 100 % on the diagonal and 0 everywhere else.


In [ ]:
"""
Evaluate the trained model on the held-out test set.

Outputs
-------
- Overall test accuracy and loss
- Per-class precision, recall, F1 (classification_report)
- Confusion matrix heatmap
"""

# ── Test loss and accuracy ─────────────────────────────────────────────────────
# model.evaluate() runs a forward pass on X_test and computes the loss and
# accuracy.  No gradient computation happens — this is inference only.
# verbose=0 suppresses the progress bar for a cleaner output.
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f'Test accuracy : {test_acc:.4f}  ({test_acc * 100:.2f}%)')
print(f'Test loss     : {test_loss:.4f}')

# ── Predictions ───────────────────────────────────────────────────────────────
# model.predict() returns the softmax probability vector for each test sample.
# Shape: (N_test, NUM_CLASSES) — one row per sample, one column per class.
y_pred_proba = model.predict(X_test, verbose=0)          # (N, NUM_CLASSES)

# np.argmax picks the column (class index) with the highest probability for
# each sample — converting the soft probability vector back to a hard class label.
# This is the argmax of the softmax output, as described in your lectures.
y_pred_int   = np.argmax(y_pred_proba, axis=1)           # predicted class indices

# ── Classification report ──────────────────────────────────────────────────────
# classification_report() computes precision, recall, and F1 for each class —
# the same metrics covered in your COMP3308 lectures:
#
#   Precision = TP / (TP + FP)  — of all samples predicted as class C, how many
#               were actually class C?  (measures false alarm rate)
#
#   Recall    = TP / (TP + FN)  — of all actual class-C samples, how many did
#               the model correctly identify?  (measures miss rate)
#
#   F1 score  = 2 * (P * R) / (P + R)  — harmonic mean of precision and recall;
#               useful when the two metrics need to be balanced.
print('\n' + '=' * 60)
print('Classification Report')
print('=' * 60)
report_str = classification_report(
    y_test_int, y_pred_int, target_names=CLASS_NAMES, zero_division=0
)
print(report_str)

# Store the report as a dictionary so it can be saved to JSON in Cell 10.
report_dict = classification_report(
    y_test_int, y_pred_int, target_names=CLASS_NAMES,
    output_dict=True, zero_division=0
)

# ── Confusion matrix ───────────────────────────────────────────────────────────
# The confusion matrix is a square grid where:
#   - row i = samples whose TRUE label is class i
#   - column j = samples that the model PREDICTED as class j
#   - diagonal cells = correct predictions
#   - off-diagonal cells = mistakes (e.g. a Rock gesture predicted as Fist)
# From your COMP3308 lectures: reading the confusion matrix tells you not just
# *how often* the model is wrong, but *which classes it confuses*.
cm = confusion_matrix(y_test_int, y_pred_int)

# Normalise to percentages (row-wise) so that class size differences don't skew
# the colour scale.  After normalisation, each row sums to 100 %.
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES + 2), max(6, NUM_CLASSES)))

# sns.heatmap draws the confusion matrix as a colour-coded grid.
# annot=True prints the numeric value inside each cell.
# fmt='.1f' formats numbers to one decimal place.
# cmap='Blues' uses a blue colour scale (darker = higher %).
sns.heatmap(
    cm_norm,
    annot      = True,
    fmt        = '.1f',
    cmap       = 'Blues',
    xticklabels = CLASS_NAMES,
    yticklabels = CLASS_NAMES,
    linewidths  = 0.5,
    cbar_kws   = {'label': 'Recall (%)'},  # colour bar label: each row is recall %
    ax         = ax,
)

ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title(f'Confusion Matrix — Test Accuracy {test_acc * 100:.2f}%',
             fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')   # rotate x-axis labels so they don't overlap
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'results/confusion_matrix_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Confusion matrix saved to results/confusion_matrix_{TIMESTAMP}.png')


---
## Cell 10 — Save Model and Results

This cell persists two artefacts for future use:

1. **`saved_models/lstm_<timestamp>.keras`** — the full trained model (weights +
   architecture).  It can be loaded later with `keras.models.load_model()` to
   make predictions without retraining.
2. **`results/lstm_<timestamp>.json`** — a JSON file recording every
   hyperparameter from CONFIG plus the final test accuracy, test loss, per-class
   classification report, and the full loss/accuracy history.  This makes it easy
   to compare multiple experimental runs side-by-side without re-running the
   notebook.


In [ ]:
import os, json
from datetime import datetime

# ─── Ensure output directories exist ─────────────────────────────────────────
os.makedirs('saved_models', exist_ok=True)  # created in Cell 6, but safe to repeat
os.makedirs('results',      exist_ok=True)

# ─── Save model weights and architecture ─────────────────────────────────────
# The .keras format saves the complete model: architecture, weights, and
# compile settings (optimiser state).  You can reload it later with:
#   model = keras.models.load_model('saved_models/lstm_<timestamp>.keras')
# and immediately call model.predict() on new glove recordings without retraining.
model_path = f'saved_models/lstm_{TIMESTAMP}.keras'
model.save(model_path)
print(f'Model saved to {model_path}')

# ─── Build a results dictionary ───────────────────────────────────────────────
# Collecting everything in one dictionary before serialising to JSON keeps the
# experiment record self-contained — you can open the JSON months later and know
# exactly which settings produced these numbers.
results = {
    'model':        'lstm',
    'timestamp':    TIMESTAMP,
    # --- Hyperparameters from CONFIG (Cell 3) ---
    'config': {
        'filter_type':     FILTER_TYPE,
        'normalization':   NORMALIZATION,
        'target_len':      TARGET_LEN,
        'feature_count':   NUM_FEATURES,
        'lstm_units':      LSTM_UNITS,      # e.g. [128, 64]
        'dropout_rate':    DROPOUT_RATE,
        'dense_units':     DENSE_UNITS,
        'epochs':          EPOCHS,
        'batch_size':      BATCH_SIZE,
        'learning_rate':   LEARNING_RATE,
    },
    # --- Test-set performance ---
    'test_accuracy':         float(test_acc),   # cast to Python float for JSON serialisation
    'test_loss':             float(test_loss),
    # --- Per-class metrics (precision, recall, F1 from Cell 9) ---
    'classification_report': report_dict,
    # --- Training curves (loss and accuracy over epochs) ---
    'history': {
        'loss':         history.history['loss'],
        'val_loss':     history.history['val_loss'],
        'accuracy':     history.history['accuracy'],
        'val_accuracy': history.history['val_accuracy'],
    },
}

# ─── Write JSON to disk ───────────────────────────────────────────────────────
# indent=2 makes the file human-readable (each key on its own line, indented).
results_path = f'results/lstm_{TIMESTAMP}.json'
with open(results_path, 'w') as fp:
    json.dump(results, fp, indent=2)
print(f'Results saved to {results_path}')


---
## Cell 11 — Inference Example *(optional)*

Load a **single** CSV trial, run it through the identical preprocessing pipeline
used during training, and print the predicted gesture label with a per-class
confidence breakdown.  This cell demonstrates how to deploy the trained model on
new recordings.

Set `INFERENCE_CSV` to the path of any valid glove CSV file.  If the file does
not exist the cell prints a skip message and exits gracefully.


In [ ]:
"""
Single-sample inference demo.

Steps
-----
1. Load the CSV with load_csv(), selecting only the feature columns used
   during training.
2. Apply the same temporal filter as training.
3. Resample to TARGET_LEN time steps.
4. Normalise with the same method as training.
5. Add a batch dimension and call model.predict().
6. Print the predicted class label and confidence scores.
"""

from utils.data_loader import (
    load_csv,
    preprocess_dataframe,
    resample_to_length,
    normalize,
)

# ─── Set the path to the CSV you want to run inference on ─────────────────────
# Change this to any gesture CSV file recorded with the glove.
INFERENCE_CSV = '../data/example_gesture/trial_001.csv'  # <-- edit this path

# ─── Check if file exists before proceeding ───────────────────────────────────
# Graceful skip prevents a FileNotFoundError when the demo path does not exist.
if not Path(INFERENCE_CSV).exists():
    print(f'[SKIP] Inference CSV not found: {INFERENCE_CSV}')
    print('Set INFERENCE_CSV to a valid path to run this cell.')
else:
    # 1. Load the CSV and keep only the columns used during training.
    #    exclude_quat=True drops quaternion columns (qw, qx, qy, qz) which are
    #    excluded from training data (they are numerically unstable).
    df_infer = load_csv(INFERENCE_CSV, feature_cols=feature_cols, exclude_quat=True)

    # Warn if any expected columns are absent in this recording.
    # Missing columns would cause a shape mismatch when fed to the model.
    loaded_cols   = df_infer.columns.tolist()
    missing_infer = [c for c in feature_cols if c not in loaded_cols]
    if missing_infer:
        print(f'WARNING: {len(missing_infer)} expected columns absent in inference CSV.')

    # 2. Apply the same temporal filter that was used during training.
    #    Skipped when FILTER_TYPE == 'none' (the default).
    if FILTER_TYPE != 'none':
        df_infer = preprocess_dataframe(df_infer, filter_type=FILTER_TYPE, fs=FS_HZ)

    # 3. Resample to TARGET_LEN timesteps.
    #    A real glove recording may have a slightly different row count than the
    #    training data (e.g. if the trial was started/stopped slightly differently).
    #    resample_to_length() stretches or compresses the time axis to exactly
    #    TARGET_LEN rows using linear interpolation.
    arr_infer = df_infer.values.astype(np.float32)
    arr_infer = resample_to_length(arr_infer, TARGET_LEN)       # → (TARGET_LEN, C)

    # 4. Normalise using the same method as training.
    #    CRITICAL: we must apply identical preprocessing at inference time.
    #    If the model was trained on min-max normalised data, raw (unnormalised)
    #    data at inference would look completely different and predictions would
    #    be garbage — this is a common real-world deployment mistake.
    arr_infer = arr_infer[np.newaxis, ...]                      # add batch dim → (1, T, C)
    if NORMALIZATION != 'none':
        arr_infer = normalize(arr_infer, method=NORMALIZATION, per_sample=True)

    # 5. Run the model on this single sample.
    #    model.predict() returns shape (1, NUM_CLASSES); we take [0] to get the
    #    (NUM_CLASSES,) probability vector for our one sample.
    proba      = model.predict(arr_infer, verbose=0)[0]         # (NUM_CLASSES,)
    pred_idx   = np.argmax(proba)      # index of the highest-probability class
    pred_label = CLASS_NAMES[pred_idx] # human-readable gesture name
    confidence = proba[pred_idx] * 100 # percentage confidence in the top prediction

    # 6. Report prediction and per-class probabilities.
    print(f'Input file   : {INFERENCE_CSV}')
    print(f'Predicted    : {pred_label}  (confidence {confidence:.1f}%)')
    print('\nPer-class probabilities:')
    # Sort by probability descending so the most likely class appears first.
    for cls, p in sorted(zip(CLASS_NAMES, proba), key=lambda t: -t[1]):
        bar = '█' * int(p * 30)   # ASCII progress bar proportional to probability
        print(f'  {cls:<30} {p * 100:5.1f}%  {bar}')
